Here is the documentation for the **Master Environment Setup** block. This script is the foundation of your Kaggle workflow, ensuring a clean, conflict-free environment for PyTorch training.

### 1. Master Environment Setup
**Purpose:**
This cell initializes the Kaggle Virtual Machine (VM). It performs three critical engineering tasks:
1.  **Noise Suppression:** Silences verbose C++ logs and Python warnings to ensure the output console remains readable.
2.  **Conflict Resolution:** Uninstalls TensorFlow (which comes pre-installed) to prevent it from fighting with PyTorch for GPU resources.
3.  **Dependency Management & Code Retrieval:** Installs specific library versions to fix known Kaggle bugs (like the TensorBoard/Protobuf crash) and clones your latest code from GitHub.

**Prerequisites:**
*   None. This should be the **very first cell** you run in your notebook.

In [ ]:
import os
import shutil
import warnings

# --- 1. SILENCE WARNINGS ---
# Hide TensorFlow/C++ low-level logs (avoids "Unable to register cuDNN" errors)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
# Hide Python Deprecation warnings (avoids "FutureWarning: torch.amp")
warnings.filterwarnings('ignore')

print("✅ Environment Silenced.")

# --- 2. SYSTEM CLEANUP ---
print("⚙️ Setting up Environment (This takes ~45 seconds)...")

# Uninstall TensorFlow to prevent GPU resource conflict with PyTorch
os.system("pip uninstall -y tensorflow tensorflow-cpu > /dev/null 2>&1")

# Install specific versions of TensorBoard and Protobuf to fix the "Red Wall" error
# Install LPIPS for the perceptual loss function (used in Stage 1)
os.system("pip install 'tensorboard>=2.10' 'protobuf<4' lpips > /dev/null 2>&1")

# --- 3. CLONE REPOSITORY ---
# TODO: Replace with your Deshadow GitHub repo URL
REPO_URL = "https://github.com/walidmoustafa2077/Deshadow.git"
REPO_DIR = "/kaggle/working/Deshadow"

# Clean start: Remove directory if it exists to avoid git merge errors
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    
print(f"📥 Cloning Repository to {REPO_DIR}...")
os.system(f"git clone {REPO_URL} {REPO_DIR}")

print("✅ Setup Complete. Ready for Config.")

**Technical Explanation:**
*   **`os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'`**: Even if you don't import TensorFlow, its background processes can spam the console with C++ warnings when they detect NVIDIA drivers. This forces them to be silent.
*   **`protobuf<4`**: Kaggle frequently updates its libraries. A recent update to `protobuf` broke compatibility with `tensorboard`, causing the "Red Wall" of errors. Forcing version `<4` creates a stable, compatible environment.
*   **`shutil.rmtree`**: This makes the script **Idempotent**. You can run this cell multiple times (e.g., after a restart) without it crashing because the folder already exists.

### 2. Configuration Patching & Optimization
**Purpose:**
This script dynamically modifies the local `src/config.py` to adapt the training pipeline for the Kaggle Cloud environment. It connects the high-resolution and low-resolution datasets, links the pre-trained Stage 1 model (required for Stage 2), and tunes hyperparameters (Batch Size/Epochs) to maximize the usage of the **Tesla P100 (16GB VRAM)** without crashing.

**Prerequisites:**
*   Repository must be cloned (Step 1).
*   Datasets must be added to the Kaggle session (`mixed-shadow-dataset`).

In [ ]:
import os
import sys

# --- 1. ENVIRONMENT CONFIGURATION ---
# Path to the High-Resolution Dataset (Read-Only) - add to Kaggle session
KAGGLE_DATASET_ROOT = "/kaggle/input/mixed-shadow-dataset/Mixed_Shadow_Dataset_1024x768"
KAGGLE_LOW_RES_ROOT = "/kaggle/input/mixed-shadow-dataset/Mixed_Shadow_Dataset_256x192"

# Path to the Pre-trained Stage 1 Model (Uploaded Dataset, optional)
STAGE1_CHECKPOINT = "/kaggle/input/deshadow/pytorch/default/1/ioanet_final.pth"

# Writable Directory for Logs & Checkpoints
WORKING_DIR = "/kaggle/working" 

# Ensure we are in the correct repository directory
# %cd is a Jupyter Magic command
%cd /kaggle/working/Deshadow

# --- 2. PATCH src/config.py ---
# Deshadow uses a Python config module (src/config.py), not config.yaml.
# We override the dataset paths and output dirs at runtime.
import src.config as cfg

# A. Dataset Connection
if os.path.exists(KAGGLE_DATASET_ROOT):
    cfg.DATA_ROOT = KAGGLE_DATASET_ROOT
    print(f"✅ High-res dataset found: {KAGGLE_DATASET_ROOT}")
else:
    print(f"❌ WARNING: High-res dataset not found at {KAGGLE_DATASET_ROOT}")

if os.path.exists(KAGGLE_LOW_RES_ROOT):
    cfg.LOW_RES_DATA_ROOT = KAGGLE_LOW_RES_ROOT
    print(f"✅ Low-res dataset found: {KAGGLE_LOW_RES_ROOT}")
else:
    print(f"❌ WARNING: Low-res dataset not found at {KAGGLE_LOW_RES_ROOT}")

# B. Link Stage 1 Checkpoint (Critical for Stage 2)
if os.path.exists(STAGE1_CHECKPOINT):
    cfg.STAGE1_CKPT = STAGE1_CHECKPOINT
    print(f"✅ Linked Stage 1 Model: {STAGE1_CHECKPOINT}")
else:
    print(f"⚠️ Stage 1 checkpoint not found at {STAGE1_CHECKPOINT}")
    print("   Will use local checkpoint after Stage 1 training.")

# C. Redirect Outputs to Writable Storage
# Kaggle only allows writing to /kaggle/working
cfg.CHECKPOINT_DIR = os.path.join(WORKING_DIR, "checkpoints")
cfg.LOG_DIR = os.path.join(WORKING_DIR, "logs")
cfg.SAMPLE_DIR = os.path.join(WORKING_DIR, "samples")

# Create output directories immediately
os.makedirs(cfg.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(cfg.LOG_DIR, exist_ok=True)
os.makedirs(cfg.SAMPLE_DIR, exist_ok=True)

# D. Stage 1 Epochs (tune for Kaggle time limits)
cfg.STAGE1_EPOCHS = 250

print("🚀 Config patched for Kaggle.")

### Technical Explanation
1.  **Dynamic Pathing:** Since Kaggle mounts inputs in specific read-only directories (`/kaggle/input`), we cannot use relative paths. This script injects the absolute paths into the `src/config.py` module.
2.  **Safety Checks:** The script uses `os.path.exists` to verify the Stage 1 model exists before linking it. This prevents the training script from crashing with a confusing `FileNotFoundError` later on.
3.  **VRAM Optimization:**
    *   **Stage 1** uses small images (256x192), so we boost batch size to `32` to saturate the GPU.
    *   **Stage 2** uses large images (1024x768). A single image takes ~1.5GB VRAM during backpropagation. We set batch size to `4` to stay within the 16GB limit of the Tesla P100.

### 3. Data Verification (Dataset Structure)
**Purpose:**
Deshadow's `ShadowRemovalDataset` reads the `input/`, `mask/`, and `target/` folders directly and resizes images on the fly to the target resolutions. No separate preprocessing script is required. This cell verifies the dataset structure is correct before training.

**Prerequisites:**
*   The `mixed-shadow-dataset` must be added to the Kaggle session.
*   The dataset must have the `train/{input,mask,target}/` structure.

**Note:** If you experience a CPU bottleneck (GPU idle while resizing), you can pre-resize the dataset to `/kaggle/working` for faster loading. The dataset loader already handles resizing, so this is optional.

In [ ]:
# 1. Verify dataset structure
# Deshadow's ShadowRemovalDataset reads input/, mask/, target/ folders directly.
# No separate preprocessing script is needed - the dataset loader resizes on the fly.
import os

def check_dataset(root, label):
    """Verify the Mixed_Shadow_Dataset structure exists."""
    if not os.path.exists(root):
        print(f"❌ {label}: not found at {root}")
        return False
    train_dir = os.path.join(root, "train")
    for sub in ["input", "mask", "target"]:
        p = os.path.join(train_dir, sub)
        if not os.path.exists(p):
            print(f"❌ {label}: missing {sub}/ folder")
            return False
    n = len(os.listdir(os.path.join(train_dir, "input")))
    print(f"✅ {label}: structure OK, {n} train images")
    return True

check_dataset("/kaggle/input/mixed-shadow-dataset/Mixed_Shadow_Dataset_256x192", "Stage 1 (low-res)")
check_dataset("/kaggle/input/mixed-shadow-dataset/Mixed_Shadow_Dataset_1024x768", "Stage 2 (high-res)")

# Note: For faster training, you can pre-resize the dataset to /kaggle/working.
# The dataset loader already resizes to LOW_RES/HIGH_RES, so this is optional.
print("✅ Dataset verification complete.")

**Technical Explanation:**
1.  **Why verify?** The dataset must have the correct `train/{input,mask,target}/` structure for `ShadowRemovalDataset` to load it. This cell catches missing folders early.
2.  **Why two versions?** Stage 1 needs small images (`256x192`) to learn global structure quickly. Stage 2 needs large images (`1024x768`) to learn fine details. The `Mixed_Shadow_Dataset` provides both versions.
3.  **Optional caching:** If resizing on the fly is slow, you can pre-resize the dataset to `/kaggle/working/fast_data` for faster loading. This is optional since the loader already handles resizing.

### 4. Dataset Path Verification
**Purpose:**
This script verifies that both the low-res (Stage 1) and high-res (Stage 2) dataset paths are accessible before training. Deshadow's dataset loader reads directly from the Kaggle input mount, so no config file patching is needed here.

**Prerequisites:**
*   The `mixed-shadow-dataset` must be added to the Kaggle session.

In [ ]:
import os

# --- PATHS ---
# Deshadow's dataset loader reads directly from the Kaggle input mount.
# No config.yaml patching is needed - the paths are set in the config cell.
LOW_RES_ROOT = "/kaggle/input/mixed-shadow-dataset/Mixed_Shadow_Dataset_256x192"
HIGH_RES_ROOT = "/kaggle/input/mixed-shadow-dataset/Mixed_Shadow_Dataset_1024x768"

# --- LOGIC ---
print("🔄 Verifying dataset paths for training...")

if os.path.exists(LOW_RES_ROOT):
    print(f"✅ Stage 1 (low-res) ready: {LOW_RES_ROOT}")
else:
    print(f"❌ Stage 1 dataset not found: {LOW_RES_ROOT}")

if os.path.exists(HIGH_RES_ROOT):
    print(f"✅ Stage 2 (high-res) ready: {HIGH_RES_ROOT}")
else:
    print(f"❌ Stage 2 dataset not found: {HIGH_RES_ROOT}")

print("🚀 Ready to train.")

**Technical Explanation:**
*   **The Check:** This verifies the dataset paths are correct before launching training. If a path is wrong, it's caught here rather than crashing mid-training.
*   **No config patching needed:** Deshadow's `ShadowRemovalDataset` reads directly from the Kaggle input mount using the paths set in the config cell (Step 2).

### 5. Executing the Training Workflow
**Purpose:**
This is the final execution step. It switches the working directory to the repository root and launches the PyTorch training process. It employs specific command-line flags to suppress non-critical warnings, ensuring the output log remains clean and readable (showing only the Progress Bar and Metrics).

**Prerequisites:**
*   Step 1 (Environment), Step 2 (Config), and Step 3 (Data) must be complete.
*   Run **Stage 1** first (trains IOANet at low-res). Then run **Stage 2** (trains the upsampler at high-res) using the Stage 1 checkpoint.

In [ ]:
# 1. Switch Context
# Move into the repository where the training scripts are located
%cd /kaggle/working/Deshadow

# 2. Launch Stage 1 Training (IOANet low-res core)
print("🚀 Starting Stage 1 Training (IOANet @ 256x192)...")

# Command Explanation:
# python -W ignore  -> Suppresses "FutureWarning" and Deprecation warnings
# train_stage1.py   -> Trains the IOANet shadow removal core
# --data_root       -> Low-res dataset (256x192)
# --epochs          -> Number of epochs (tuned for Kaggle time limits)
# --batch_size      -> Low-res images are small, so we can use a larger batch
!python -W ignore src/train_stage1.py \
  --data_root /kaggle/input/mixed-shadow-dataset/Mixed_Shadow_Dataset_256x192 \
  --epochs 250 \
  --batch_size 32 \
  --save_dir /kaggle/working/checkpoints

# --- For Stage 2 (after Stage 1 completes), uncomment: ---
# !python -W ignore src/train_stage2.py \
#   --data_root /kaggle/input/mixed-shadow-dataset/Mixed_Shadow_Dataset_1024x768 \
#   --ioanet_ckpt /kaggle/working/checkpoints/ioanet_final.pth \
#   --epochs 200 \
#   --batch_size 4 \
#   --save_dir /kaggle/working/checkpoints

**Technical Explanation:**
*   **`%cd` vs `cd`:** In Jupyter/Kaggle, standard `cd` only changes the directory for that specific line. `%cd` is a "Magic Command" that changes the current working directory for the **entire notebook session**. This is critical so that `train.py` can find the `src/` and `configs/` folders relative to itself.
*   **`-W ignore`:** The PyTorch ecosystem often emits warnings about future version changes (e.g., `torch.cuda.amp` vs `torch.amp`). These are informative but clog the logs. This flag filters them out so you don't miss important metric updates like Loss or PSNR.

### 6. Artifact Extraction (Save Your Work)
**Purpose:**
Kaggle Notebooks are ephemeral (temporary). If the session ends or disconnects, all files in `/kaggle/working` are deleted. This script aggregates your training results (Model Weights and Sample Images) into downloadable ZIP archives and generates clickable links. It is the bridge between the Cloud VM and your local machine.

**Prerequisites:**
*   Training (Stage 1 or Stage 2) must have run for at least one epoch.
*   The folders `checkpoints` and `samples` must exist in `/kaggle/working`.

**Technical Explanation:**
1.  **`os.chdir("/kaggle/working")`**: This is the most critical line. During training, we used `%cd` to enter the repo folder. However, `FileLink` generates URLs relative to the notebook's root. By moving back to the root, we ensure the download link points to the correct location.
2.  **`shutil.make_archive`**: A high-level Python function that handles ZIP compression efficiently without needing external shell commands.
3.  **`FileLink`**: A specific Jupyter/IPython widget that renders a local file path as a clickable HTTP download link in the browser, allowing you to bypass the Kaggle UI file browser.

In [ ]:
import os
import shutil
import zipfile
from IPython.display import FileLink, display

# --- 1. CLEANUP ---
# Remove any old zip files to prevent confusion or disk space issues
!rm -f /kaggle/working/*.zip
print("✅ Cleanup of old archives complete.")

# --- 2. CONTEXT SWITCH ---
# CRITICAL: Switch to the root directory so FileLink can find the files correctly.
os.chdir("/kaggle/working")
print(f"📍 Current Working Directory: {os.getcwd()}")

# --- CONFIGURATION ---
SOURCE_CHECKPOINTS = "checkpoints"  # Relative to /kaggle/working
SOURCE_SAMPLES = "samples"          # Relative to /kaggle/working

# --- LOGIC ---

def zip_and_link(folder_name, zip_name):
    """Zips an entire directory."""
    if not os.path.exists(folder_name):
        print(f"⚠️  Skipping: '{folder_name}' not found.")
        return

    print(f"📦 Zipping directory '{folder_name}' to '{zip_name}.zip'...")
    
    shutil.make_archive(zip_name, 'zip', folder_name)
    
    if os.path.exists(f"{zip_name}.zip"):
        print(f"✅ Created: {zip_name}.zip")
        display(FileLink(f"{zip_name}.zip"))
    else:
        print(f"❌ Error: Failed to create {zip_name}.zip")

def zip_best_only():
    """
    Searches recursively for the final model checkpoints and zips ONLY those.
    Deshadow saves ioanet_final.pth (Stage 1) and upsampler_final.pth (Stage 2).
    """
    target_files = ["ioanet_final.pth", "upsampler_final.pth"]
    output_zip = "best_model_only.zip"
    files_to_zip = []

    if os.path.exists(SOURCE_CHECKPOINTS):
        for root, dirs, files in os.walk(SOURCE_CHECKPOINTS):
            for f in files:
                if f in target_files:
                    full_path = os.path.join(root, f)
                    files_to_zip.append(full_path)
    
    if not files_to_zip:
        print(f"⚠️  No final model checkpoints found to zip independently.")
        return

    print(f"💎 Creating lightweight '{output_zip}'...")
    
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for file_path in files_to_zip:
            arcname = os.path.relpath(file_path, SOURCE_CHECKPOINTS)
            zf.write(file_path, arcname)
            print(f"   -> Added: {arcname}")

    display(FileLink(output_zip))

# --- EXECUTION ---

# 1. Zip Samples (Visuals)
zip_and_link(SOURCE_SAMPLES, "my_samples")

print("-" * 30)

# 2. Zip ONLY Best Models (Lightweight)
zip_best_only()

print("-" * 30)

# 3. Zip ALL Checkpoints (Heavy Backup - includes intermediate epochs)
zip_and_link(SOURCE_CHECKPOINTS, "full_checkpoint_backup")